# STAC + Xarray + NDWI + Titiler -> island detection in the Gulf of Morbihan 

This Notebook showcases using [jupyter-tiler](https://github.com/geojupyter/jupyter-tiler) to connect to a STAC catalog and write an island detection algorithm then visualize the result on an ipyleaflet map.

The tiles are computed on demand, so panning and zooming the map triggers computing new areas of the map.

In [ ]:
from ipyleaflet import Map

latitude = 47.581515
longitude = -2.803342

m = Map(center=[latitude, longitude], zoom=13)
m

In [ ]:
import numpy as np
from rio_tiler.models import ImageData
from xarray import DataArray

from jupyter_tiler.titiler import add_stac_array


def ndwi_process(data: DataArray) -> ImageData:
    if "time" in data.dims:
        # pick one scene OR use median over time; choose one
        data = data.isel(time=0)
        # data = data.median(dim="time", skipna=True)

    green = data.sel(band="green").data.astype(np.float32)
    nir = data.sel(band="nir").data.astype(np.float32)

    ndwi = (green - nir) / (green + nir + 1e-6)
    ndwi = np.asarray(ndwi)  # drop masked-array dimensional surprises
    ndwi = np.nan_to_num(ndwi, nan=-1.0, posinf=1.0, neginf=-1.0)

    ndwi_01 = np.clip((ndwi + 1.0) / 2.0, 0.0, 1.0)
    pixels = (ndwi_01 * 255).astype(np.uint8)

    return ImageData(pixels[np.newaxis, :, :])


url = await add_stac_array(
    stac_url="https://earth-search.aws.element84.com/v1",
    array_to_image=ndwi_process,
    collection_id="sentinel-2-l2a",
    assets=["green", "nir"],
    # Perf parameters
    max_items=5,
    # Any other kwarg is treated as a STAC search parameter, see `ItemSearch`
    # API from `pystac_client`.
    datetime="2024-06-01/2024-09-30",
    query={"eo:cloud_cover": {"lt": 20}},
)

In [ ]:
url

In [ ]:
from ipyleaflet import TileLayer

m.add_layer(TileLayer(url=url))